In [ ]:
# Cài đặt thư viện transformers để tải mô hình ESM-2
!pip install transformers datasets -q
!pip install -q transformers accelerate lightgbm

In [ ]:
# ============================================================
# PHASE 1.2B — ESM-2 SLIDING-WINDOW EMBEDDING ON COLAB GPU
# ============================================================

import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("--- Hardware Check ---")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

--- Hardware Check ---
Torch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# GOOGLE DRIVE PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

SPLIT_DIR = PROJECT_DIR / "model" / "phase1_protein_baseline" / "splits"

BASE_DIR = PROJECT_DIR / "model" / "phase1_2_esm2_sliding_window_embedding"

EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [BASE_DIR, EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Split dir:", SPLIT_DIR)
print("Output dir:", BASE_DIR)

assert SPLIT_DIR.exists(), f"Split directory not found: {SPLIT_DIR}"

Split dir: /content/drive/MyDrive/Project_Protein/model/phase1_protein_baseline/splits
Output dir: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding


In [ ]:
# ============================================================
# LOAD SAVED SPLITS
# ============================================================

train_path = SPLIT_DIR / "train_protein_v1.csv"
val_path = SPLIT_DIR / "val_protein_v1.csv"
test_path = SPLIT_DIR / "test_protein_v1.csv"

assert train_path.exists(), train_path
assert val_path.exists(), val_path
assert test_path.exists(), test_path

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train labels:
label
0    637
1    637
Name: count, dtype: int64

Validation labels:
label
1    137
0    136
Name: count, dtype: int64

Test labels:
label
0    137
1    136
Name: count, dtype: int64


In [ ]:
# ============================================================
# CLEAN SEQUENCES
# Keep only 20 standard amino acids
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test length summary:")
display(test_df["sequence_clean_length"].describe())

Train length summary:


,sequence_clean_length
count,1274.000000
mean,744.512559
std,643.266085
min,41.000000
25%,354.000000
50%,555.000000
75%,926.000000
max,7388.000000


Validation length summary:


,sequence_clean_length
count,273.000000
mean,869.728938
std,2114.423720
min,56.000000
25%,370.000000
50%,576.000000
75%,977.000000
max,34350.000000


Test length summary:


,sequence_clean_length
count,273.000000
mean,774.432234
std,689.578849
min,51.000000
25%,326.000000
50%,574.000000
75%,1035.000000
max,4861.000000


In [ ]:
# ============================================================
# LOAD ESM-2 MODEL
# ============================================================

ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

print("Loading tokenizer:", ESM_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)

print("Loading model:", ESM_MODEL_NAME)

if device.type == "cuda":
    esm_model = AutoModel.from_pretrained(
        ESM_MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME)

esm_model.to(device)
esm_model.eval()

print("Model loaded.")
print("Device:", device)

Loading tokenizer: facebook/esm2_t6_8M_UR50D


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Device: cuda


In [ ]:
# ============================================================
# SLIDING-WINDOW CONFIG
# ============================================================

WINDOW_SIZE = 1022
STRIDE = 1022

WINDOW_BATCH_SIZE = 16     # If OOM, reduce to 8 or 4
SAVE_EVERY = 50            # checkpoint every 50 proteins

REPRESENTATION_NAME = "ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022"

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Window batch size:", WINDOW_BATCH_SIZE)
print("Representation:", REPRESENTATION_NAME)

Window size: 1022
Stride: 1022
Window batch size: 16
Representation: ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [ ]:
# ============================================================
# WINDOWING FUNCTION
# ============================================================

def split_sequence_into_windows(seq, window_size=1022, stride=1022):
    seq = str(seq)

    if len(seq) == 0:
        raise ValueError("Empty sequence after cleaning.")

    if len(seq) <= window_size:
        return [seq]

    windows = []

    start = 0
    while start < len(seq):
        end = start + window_size
        window = seq[start:end]

        if len(window) > 0:
            windows.append(window)

        start += stride

    return windows


# Quick test
example_seq = "A" * 2500
example_windows = split_sequence_into_windows(example_seq, WINDOW_SIZE, STRIDE)

print("Example number of windows:", len(example_windows))
print("Window lengths:", [len(w) for w in example_windows])

Example number of windows: 3
Window lengths: [1022, 1022, 456]


In [ ]:
# ============================================================
# BATCH WINDOW EMBEDDING
# ============================================================

@torch.no_grad()
def embed_window_batch(
    window_seqs,
    tokenizer,
    model,
    device
):
    """
    Embed a batch of protein windows.

    Returns:
        batch_embeddings: np.ndarray, shape [n_windows, hidden_dim]
    """
    encoded = tokenizer(
        window_seqs,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
        truncation=True
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    if device.type == "cuda":
        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(**encoded)
    else:
        outputs = model(**encoded)

    hidden = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]

    batch_embeddings = []

    for i in range(hidden.shape[0]):
        mask = attention_mask[i].bool()
        valid_positions = torch.where(mask)[0]

        # Exclude special tokens if possible
        if len(valid_positions) > 2:
            aa_positions = valid_positions[1:-1]
        else:
            aa_positions = valid_positions

        aa_hidden = hidden[i, aa_positions, :]
        emb = aa_hidden.mean(dim=0)

        batch_embeddings.append(emb.detach().float().cpu().numpy())

    batch_embeddings = np.vstack(batch_embeddings)

    del encoded, outputs, hidden, attention_mask

    return batch_embeddings

In [ ]:
# ============================================================
# EMBED ONE FULL PROTEIN USING SLIDING WINDOWS
# ============================================================

@torch.no_grad()
def embed_one_sequence_sliding_window_batched(
    seq,
    tokenizer,
    model,
    device,
    window_size=1022,
    stride=1022,
    window_batch_size=16
):
    """
    Full protein embedding:
    1. Split sequence into windows
    2. Embed windows in batches
    3. Average all window embeddings into one protein embedding
    """
    windows = split_sequence_into_windows(
        seq,
        window_size=window_size,
        stride=stride
    )

    all_window_embeddings = []

    for start in range(0, len(windows), window_batch_size):
        batch_windows = windows[start:start + window_batch_size]

        batch_emb = embed_window_batch(
            window_seqs=batch_windows,
            tokenizer=tokenizer,
            model=model,
            device=device
        )

        all_window_embeddings.append(batch_emb)

    all_window_embeddings = np.vstack(all_window_embeddings)

    protein_embedding = all_window_embeddings.mean(axis=0)

    return protein_embedding, len(windows)

In [ ]:
# ============================================================
# DEBUG ONE SEQUENCE
# ============================================================

debug_seq = train_df.iloc[0]["sequence_clean"]

debug_emb, debug_n_windows = embed_one_sequence_sliding_window_batched(
    seq=debug_seq,
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE
)

print("Debug embedding shape:", debug_emb.shape)
print("Number of windows:", debug_n_windows)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

Debug embedding shape: (320,)
Number of windows: 1
GPU memory allocated: 23.56201171875 MB
GPU memory reserved: 38.0 MB


In [ ]:
# ============================================================
# ROBUST SPLIT EXTRACTION WITH RESUME
# ============================================================

def extract_sliding_embeddings_for_split_gpu(
    split_df,
    split_name,
    tokenizer,
    model,
    device,
    output_dir,
    sequence_col="sequence_clean",
    window_size=1022,
    stride=1022,
    window_batch_size=16,
    save_every=50
):
    output_dir = Path(output_dir)

    final_embedding_path = output_dir / f"esm2_sw_{split_name}_embeddings.npy"
    final_metadata_path = output_dir / f"esm2_sw_{split_name}_metadata.csv"

    partial_embedding_path = output_dir / f"esm2_sw_{split_name}_embeddings_partial.npy"
    partial_metadata_path = output_dir / f"esm2_sw_{split_name}_metadata_partial.csv"

    # If final exists, load and return
    if final_embedding_path.exists() and final_metadata_path.exists():
        print(f"[{split_name}] Final embeddings already exist. Loading...")
        embeddings = np.load(final_embedding_path)
        metadata = pd.read_csv(final_metadata_path)
        return embeddings, metadata

    split_df_reset = split_df.reset_index(drop=True)

    embeddings = []
    metadata_records = []
    start_idx = 0

    # Resume from partial if available
    if partial_embedding_path.exists() and partial_metadata_path.exists():
        print(f"[{split_name}] Partial checkpoint found. Resuming...")

        partial_embeddings = np.load(partial_embedding_path)
        partial_metadata = pd.read_csv(partial_metadata_path)

        embeddings = [partial_embeddings[i] for i in range(partial_embeddings.shape[0])]
        metadata_records = partial_metadata.to_dict("records")

        start_idx = len(metadata_records)

        print(f"[{split_name}] Resuming from index {start_idx}")

    total_windows_processed = sum(
        record.get("n_windows", 0) for record in metadata_records
    )

    start_time = time.time()

    for idx in range(start_idx, len(split_df_reset)):
        row = split_df_reset.iloc[idx]
        seq = row[sequence_col]

        try:
            emb, n_windows = embed_one_sequence_sliding_window_batched(
                seq=seq,
                tokenizer=tokenizer,
                model=model,
                device=device,
                window_size=window_size,
                stride=stride,
                window_batch_size=window_batch_size
            )

            embeddings.append(emb)
            total_windows_processed += n_windows

            metadata_records.append({
                "row_index": idx,
                "gene_id": row.get("gene_id", None),
                "gene_symbol": row.get("gene_symbol", None),
                "uniprot_accession": row.get("uniprot_accession", None),
                "label": int(row["label"]),
                "sequence_clean_length": len(seq),
                "n_windows": n_windows,
                "window_size": window_size,
                "stride": stride,
                "representation": REPRESENTATION_NAME
            })

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"CUDA OOM at {split_name} index {idx}.")
                print("Try reducing WINDOW_BATCH_SIZE.")
                torch.cuda.empty_cache()
            raise e

        except Exception as e:
            print(f"Error at {split_name} index {idx}: {e}")
            raise e

        # Checkpoint
        if (idx + 1) % save_every == 0 or (idx + 1) == len(split_df_reset):
            current_embeddings = np.vstack(embeddings)
            current_metadata = pd.DataFrame(metadata_records)

            np.save(partial_embedding_path, current_embeddings)
            current_metadata.to_csv(partial_metadata_path, index=False)

            elapsed = time.time() - start_time
            avg_windows = total_windows_processed / len(metadata_records)

            print(
                f"[{split_name}] {idx + 1}/{len(split_df_reset)} proteins "
                f"| embeddings={current_embeddings.shape} "
                f"| total_windows={total_windows_processed} "
                f"| avg_windows/protein={avg_windows:.2f} "
                f"| elapsed={elapsed/60:.2f} min"
            )

            if device.type == "cuda":
                torch.cuda.empty_cache()

    final_embeddings = np.vstack(embeddings)
    final_metadata = pd.DataFrame(metadata_records)

    np.save(final_embedding_path, final_embeddings)
    final_metadata.to_csv(final_metadata_path, index=False)

    # Keep partial files or remove them.
    # I keep them for safety. Uncomment to remove:
    # partial_embedding_path.unlink(missing_ok=True)
    # partial_metadata_path.unlink(missing_ok=True)

    print(f"[{split_name}] Final embeddings shape:", final_embeddings.shape)
    print(f"[{split_name}] Saved:", final_embedding_path)
    print(f"[{split_name}] Metadata saved:", final_metadata_path)

    return final_embeddings, final_metadata

In [ ]:
# ============================================================
# DEBUG RUN — 10 PROTEINS
# ============================================================

debug_df = train_df.head(10).copy()

X_debug_sw, meta_debug_sw = extract_sliding_embeddings_for_split_gpu(
    split_df=debug_df,
    split_name="debug10",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=5
)

print("Debug shape:", X_debug_sw.shape)
display(meta_debug_sw)

[debug10] 5/10 proteins | embeddings=(5, 320) | total_windows=5 | avg_windows/protein=1.00 | elapsed=0.00 min
[debug10] 10/10 proteins | embeddings=(10, 320) | total_windows=10 | avg_windows/protein=1.00 | elapsed=0.00 min
[debug10] Final embeddings shape: (10, 320)
[debug10] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_debug10_embeddings.npy
[debug10] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_debug10_metadata.csv
Debug shape: (10, 320)


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
5,5,ENSG00000213996,TM6SF2,Q9BZW4,1,377,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
6,6,ENSG00000164823,OSGIN2,Q9Y236,0,505,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
7,7,ENSG00000185716,MOSMO,Q8NHV5,0,167,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
8,8,ENSG00000181873,IBA57,Q5T440,0,356,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
9,9,ENSG00000164647,STEAP1,Q9UHE8,1,339,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [ ]:
# ============================================================
# EXTRACT VALIDATION SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_val_sw, meta_val_sw = extract_sliding_embeddings_for_split_gpu(
    split_df=val_df,
    split_name="val",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[val] 50/273 proteins | embeddings=(50, 320) | total_windows=70 | avg_windows/protein=1.40 | elapsed=0.02 min
[val] 100/273 proteins | embeddings=(100, 320) | total_windows=138 | avg_windows/protein=1.38 | elapsed=0.05 min
[val] 150/273 proteins | embeddings=(150, 320) | total_windows=206 | avg_windows/protein=1.37 | elapsed=0.06 min
[val] 200/273 proteins | embeddings=(200, 320) | total_windows=297 | avg_windows/protein=1.49 | elapsed=0.07 min
[val] 250/273 proteins | embeddings=(250, 320) | total_windows=357 | avg_windows/protein=1.43 | elapsed=0.09 min
[val] 273/273 proteins | embeddings=(273, 320) | total_windows=383 | avg_windows/protein=1.40 | elapsed=0.09 min
[val] Final embeddings shape: (273, 320)
[val] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_val_embeddings.npy
[val] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_val_metadata.csv


In [ ]:
# ============================================================
# EXTRACT TEST SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_test_sw, meta_test_sw = extract_sliding_embeddings_for_split_gpu(
    split_df=test_df,
    split_name="test",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[test] 50/273 proteins | embeddings=(50, 320) | total_windows=61 | avg_windows/protein=1.22 | elapsed=0.02 min
[test] 100/273 proteins | embeddings=(100, 320) | total_windows=124 | avg_windows/protein=1.24 | elapsed=0.04 min
[test] 150/273 proteins | embeddings=(150, 320) | total_windows=189 | avg_windows/protein=1.26 | elapsed=0.05 min
[test] 200/273 proteins | embeddings=(200, 320) | total_windows=263 | avg_windows/protein=1.31 | elapsed=0.07 min
[test] 250/273 proteins | embeddings=(250, 320) | total_windows=328 | avg_windows/protein=1.31 | elapsed=0.08 min
[test] 273/273 proteins | embeddings=(273, 320) | total_windows=363 | avg_windows/protein=1.33 | elapsed=0.09 min
[test] Final embeddings shape: (273, 320)
[test] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_test_embeddings.npy
[test] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/embeddings/esm2_sw_test_metadata.

In [ ]:
# ============================================================
# EXTRACT TRAIN SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_train_sw, meta_train_sw = extract_sliding_embeddings_for_split_gpu(
    split_df=train_df,
    split_name="train",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[train] 50/1274 proteins | embeddings=(50, 320) | total_windows=64 | avg_windows/protein=1.28 | elapsed=0.02 min
[train] 100/1274 proteins | embeddings=(100, 320) | total_windows=127 | avg_windows/protein=1.27 | elapsed=0.04 min
[train] 150/1274 proteins | embeddings=(150, 320) | total_windows=187 | avg_windows/protein=1.25 | elapsed=0.05 min
[train] 200/1274 proteins | embeddings=(200, 320) | total_windows=251 | avg_windows/protein=1.25 | elapsed=0.07 min
[train] 250/1274 proteins | embeddings=(250, 320) | total_windows=317 | avg_windows/protein=1.27 | elapsed=0.08 min
[train] 300/1274 proteins | embeddings=(300, 320) | total_windows=391 | avg_windows/protein=1.30 | elapsed=0.09 min
[train] 350/1274 proteins | embeddings=(350, 320) | total_windows=455 | avg_windows/protein=1.30 | elapsed=0.10 min
[train] 400/1274 proteins | embeddings=(400, 320) | total_windows=518 | avg_windows/protein=1.29 | elapsed=0.11 min
[train] 450/1274 proteins | embeddings=(450, 320) | total_windows=581 | avg

In [ ]:
# ============================================================
# LOAD SAVED EMBEDDINGS
# ============================================================

X_train_sw = np.load(EMBED_DIR / "esm2_sw_train_embeddings.npy")
X_val_sw = np.load(EMBED_DIR / "esm2_sw_val_embeddings.npy")
X_test_sw = np.load(EMBED_DIR / "esm2_sw_test_embeddings.npy")

meta_train_sw = pd.read_csv(EMBED_DIR / "esm2_sw_train_metadata.csv")
meta_val_sw = pd.read_csv(EMBED_DIR / "esm2_sw_val_metadata.csv")
meta_test_sw = pd.read_csv(EMBED_DIR / "esm2_sw_test_metadata.csv")

y_train = meta_train_sw["label"].astype(int)
y_val = meta_val_sw["label"].astype(int)
y_test = meta_test_sw["label"].astype(int)

print("X_train_sw:", X_train_sw.shape)
print("X_val_sw:", X_val_sw.shape)
print("X_test_sw:", X_test_sw.shape)

print("\nTrain labels:")
print(y_train.value_counts())

display(meta_train_sw.head())

X_train_sw: (1274, 320)
X_val_sw: (273, 320)
X_test_sw: (273, 320)

Train labels:
label
0    637
1    637
Name: count, dtype: int64


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [ ]:
# ============================================================
# WINDOW SUMMARY
# ============================================================

window_summary = []

for split_name, meta in [
    ("train", meta_train_sw),
    ("validation", meta_val_sw),
    ("test", meta_test_sw)
]:
    window_summary.append({
        "split": split_name,
        "n": len(meta),
        "mean_sequence_length": meta["sequence_clean_length"].mean(),
        "median_sequence_length": meta["sequence_clean_length"].median(),
        "max_sequence_length": meta["sequence_clean_length"].max(),
        "mean_n_windows": meta["n_windows"].mean(),
        "median_n_windows": meta["n_windows"].median(),
        "max_n_windows": meta["n_windows"].max(),
        "n_multi_window_proteins": int((meta["n_windows"] > 1).sum()),
        "pct_multi_window_proteins": (meta["n_windows"] > 1).mean() * 100
    })

window_summary_df = pd.DataFrame(window_summary)

display(window_summary_df)

window_summary_df.to_csv(
    RESULT_DIR / "esm2_sliding_window_summary_v1.csv",
    index=False
)

,split,n,mean_sequence_length,median_sequence_length,max_sequence_length,mean_n_windows,median_n_windows,max_n_windows,n_multi_window_proteins,pct_multi_window_proteins
0,train,1274,744.512559,555.0,7388,1.267661,1.0,8,263,20.643642
1,validation,273,869.728938,576.0,34350,1.402930,1.0,34,65,23.809524
2,test,273,774.432234,574.0,4861,1.329670,1.0,5,69,25.274725


In [ ]:
# ============================================================
# FREE GPU MEMORY AFTER EMBEDDING
# ============================================================

del esm_model
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [ ]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [ ]:
# ============================================================
# DOWNSTREAM MODELS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


sw_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    sw_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    sw_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("Models:")
for name in sw_models:
    print("-", name)

Models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [ ]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

sw_baseline_cv_results = []

for model_name, pipeline in sw_models.items():
    print("=" * 80)
    print("Sliding-window ESM baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train_sw,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    sw_baseline_cv_results.append(row)

sw_baseline_cv_df = pd.DataFrame(sw_baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(sw_baseline_cv_df)

sw_baseline_cv_df.to_csv(
    RESULT_DIR / "esm2_sw_baseline_cv_before_tuning_v1.csv",
    index=False
)

Sliding-window ESM baseline CV: Logistic Regression
Sliding-window ESM baseline CV: SVM RBF
Sliding-window ESM baseline CV: Random Forest
Sliding-window ESM baseline CV: LightGBM


,representation,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
2,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,Random Forest,1.000000,0.000000,0.694318,0.026275,0.638893,0.017556,0.682195,0.028573,0.268870,0.043736,0.305682
1,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,SVM RBF,0.910729,0.001929,0.684340,0.015452,0.625118,0.024190,0.672955,0.022327,0.264506,0.035660,0.226388
3,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,LightGBM,1.000000,0.000000,0.668426,0.032137,0.615070,0.025808,0.653952,0.031553,0.223072,0.062188,0.331574
0,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,Logistic Regression,0.887706,0.005985,0.623304,0.043990,0.593497,0.037517,0.619953,0.052664,0.197951,0.077617,0.264402


In [ ]:
# ============================================================
# PARAMETER GRIDS — COLAB-OPTIMIZED
# ============================================================

sw_param_grids = {
    "Logistic Regression": {
        "model__C": [0.003, 0.01, 0.03, 0.1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.1, 1, 10],
        "model__gamma": [0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    sw_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05],
        "model__num_leaves": [7, 15],
        "model__max_depth": [3, 5],
        "model__min_child_samples": [20, 50],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0.1],
        "model__reg_lambda": [1, 5]
    }
else:
    sw_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.03, 0.05],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 50]
    }

In [30]:
# ============================================================
# RUN GRIDSEARCHCV
# ============================================================

sw_grid_search_results = []
sw_best_estimators = {}

for model_name, pipeline in sw_models.items():
    print("=" * 80)
    print("Sliding-window ESM GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=sw_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_sw, y_train)

    sw_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    sw_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

sw_grid_results_df = pd.DataFrame(sw_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(sw_grid_results_df)

sw_grid_results_df.to_csv(
    RESULT_DIR / "esm2_sw_gridsearch_results_v1.csv",
    index=False
)

Sliding-window ESM GridSearchCV: Logistic Regression
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV ROC-AUC: 0.6666928552607105
Best train ROC-AUC: 0.7947932176607304
Gap: 0.12810036240001987
Best params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Sliding-window ESM GridSearchCV: SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.684340415555831
Best train ROC-AUC: 0.9107287812077718
Gap: 0.22638836565194076
Best params: {'model__C': 1, 'model__gamma': 'scale'}
Sliding-window ESM GridSearchCV: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV ROC-AUC: 0.6949317804885611
Best train ROC-AUC: 0.9979545240647205
Gap: 0.30302274357615944
Best params: {'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 0.2, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
Sliding-window ESM GridSearchCV: LightGBM
Fitting 5 folds for each of 128 candidates, totalli

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
2,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,Random Forest,0.694932,0.997955,0.303023,"{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 0.2, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}"
3,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,LightGBM,0.685333,0.994548,0.309214,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 5, 'model__min_child_samples': 50, 'model__n_estimators': 200, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 5, 'model__subsample': 0.8}"
1,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,SVM RBF,0.684340,0.910729,0.226388,"{'model__C': 1, 'model__gamma': 'scale'}"
0,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,Logistic Regression,0.666693,0.794793,0.128100,"{'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"


In [31]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

sw_tuned_eval_records = []

for model_name, model in sw_best_estimators.items():
    print("=" * 80)
    print("Evaluating:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_sw,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_sw,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["representation"] = REPRESENTATION_NAME
    val_metrics["representation"] = REPRESENTATION_NAME

    sw_tuned_eval_records.extend([train_metrics, val_metrics])

sw_tuned_eval_df = pd.DataFrame(sw_tuned_eval_records)

display(sw_tuned_eval_df)

sw_validation_comparison = sw_tuned_eval_df[
    sw_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(sw_validation_comparison)

sw_tuned_eval_df.to_csv(
    RESULT_DIR / "esm2_sw_tuned_train_validation_metrics_v1.csv",
    index=False
)

sw_validation_comparison.to_csv(
    RESULT_DIR / "esm2_sw_validation_comparison_v1.csv",
    index=False
)

Evaluating: Logistic Regression
Evaluating: SVM RBF
Evaluating: Random Forest
Evaluating: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,train,0.718995,0.723917,0.708006,0.729984,0.715873,0.785881,0.777027,0.438096,465,172,186,451,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,Logistic Regression,validation,0.644689,0.663934,0.591241,0.698529,0.625483,0.722842,0.721061,0.291417,95,41,56,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
2,SVM RBF,train,0.824961,0.829618,0.817896,0.832025,0.823715,0.905101,0.905696,0.649986,530,107,116,521,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.633700,0.650407,0.583942,0.683824,0.615385,0.691740,0.682528,0.269083,93,43,57,80,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
4,Random Forest,train,0.968603,0.964230,0.973312,0.963893,0.968750,0.995515,0.995621,0.937247,614,23,17,620,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.652015,0.647887,0.671533,0.632353,0.659498,0.713772,0.720897,0.304131,86,50,45,92,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
6,LightGBM,train,0.958399,0.956250,0.960754,0.956044,0.958496,0.992572,0.993229,0.916808,609,28,25,612,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
7,LightGBM,validation,0.659341,0.664179,0.649635,0.669118,0.656827,0.725258,0.714919,0.318804,91,45,48,89,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
7,LightGBM,validation,0.659341,0.664179,0.649635,0.669118,0.656827,0.725258,0.714919,0.318804,91,45,48,89,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,Logistic Regression,validation,0.644689,0.663934,0.591241,0.698529,0.625483,0.722842,0.721061,0.291417,95,41,56,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.652015,0.647887,0.671533,0.632353,0.659498,0.713772,0.720897,0.304131,86,50,45,92,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.633700,0.650407,0.583942,0.683824,0.615385,0.691740,0.682528,0.269083,93,43,57,80,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [32]:
# ============================================================
# SOFT VOTING
# ============================================================

sw_voting_estimators = []

for model_name, estimator in sw_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    sw_voting_estimators.append((short_name, estimator))

sw_soft_voting = VotingClassifier(
    estimators=sw_voting_estimators,
    voting="soft",
    n_jobs=-1
)

sw_soft_voting.fit(X_train_sw, y_train)

sw_voting_train_metrics = evaluate_binary_classifier(
    sw_soft_voting,
    X_train_sw,
    y_train,
    "Soft Voting",
    "train"
)

sw_voting_val_metrics = evaluate_binary_classifier(
    sw_soft_voting,
    X_val_sw,
    y_val,
    "Soft Voting",
    "validation"
)

sw_voting_train_metrics["representation"] = REPRESENTATION_NAME
sw_voting_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([sw_voting_train_metrics, sw_voting_val_metrics]))

sw_best_estimators["Soft Voting"] = sw_soft_voting

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,train,0.894035,0.899682,0.886970,0.901099,0.893281,0.962935,0.964458,0.788148,574,63,72,565,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,Soft Voting,validation,0.659341,0.666667,0.642336,0.676471,0.654275,0.728532,0.730087,0.318978,92,44,49,88,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [33]:
# ============================================================
# STACKING
# ============================================================

sw_stacking_estimators = []

for model_name, estimator in sw_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    sw_stacking_estimators.append((short_name, estimator))

sw_stacking = StackingClassifier(
    estimators=sw_stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

sw_stacking.fit(X_train_sw, y_train)

sw_stacking_train_metrics = evaluate_binary_classifier(
    sw_stacking,
    X_train_sw,
    y_train,
    "Stacking",
    "train"
)

sw_stacking_val_metrics = evaluate_binary_classifier(
    sw_stacking,
    X_val_sw,
    y_val,
    "Stacking",
    "validation"
)

sw_stacking_train_metrics["representation"] = REPRESENTATION_NAME
sw_stacking_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([sw_stacking_train_metrics, sw_stacking_val_metrics]))

sw_best_estimators["Stacking"] = sw_stacking

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,train,0.922292,0.921630,0.923077,0.921507,0.922353,0.976558,0.977532,0.844585,587,50,49,588,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,Stacking,validation,0.644689,0.649254,0.635036,0.654412,0.642066,0.725258,0.728313,0.289495,89,47,50,87,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [34]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

sw_all_eval_df = pd.concat(
    [
        sw_tuned_eval_df,
        pd.DataFrame([
            sw_voting_train_metrics,
            sw_voting_val_metrics,
            sw_stacking_train_metrics,
            sw_stacking_val_metrics
        ])
    ],
    ignore_index=True
)

sw_all_validation_results = sw_all_eval_df[
    sw_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(sw_all_validation_results)

sw_all_eval_df.to_csv(
    RESULT_DIR / "esm2_sw_all_train_validation_metrics_v1.csv",
    index=False
)

sw_all_validation_results.to_csv(
    RESULT_DIR / "esm2_sw_all_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
9,Soft Voting,validation,0.659341,0.666667,0.642336,0.676471,0.654275,0.728532,0.730087,0.318978,92,44,49,88,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
7,LightGBM,validation,0.659341,0.664179,0.649635,0.669118,0.656827,0.725258,0.714919,0.318804,91,45,48,89,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
11,Stacking,validation,0.644689,0.649254,0.635036,0.654412,0.642066,0.725258,0.728313,0.289495,89,47,50,87,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,Logistic Regression,validation,0.644689,0.663934,0.591241,0.698529,0.625483,0.722842,0.721061,0.291417,95,41,56,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.652015,0.647887,0.671533,0.632353,0.659498,0.713772,0.720897,0.304131,86,50,45,92,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.633700,0.650407,0.583942,0.683824,0.615385,0.691740,0.682528,0.269083,93,43,57,80,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [35]:
# ============================================================
# SELECT FINAL MODEL
# ============================================================

sw_best_validation_row = sw_all_validation_results.iloc[0]
final_sw_model_name = sw_best_validation_row["model"]
final_sw_model = sw_best_estimators[final_sw_model_name]

print("Final selected sliding-window ESM model:", final_sw_model_name)
display(sw_best_validation_row)

pd.DataFrame([sw_best_validation_row]).to_csv(
    RESULT_DIR / "esm2_sw_final_model_validation_summary_v1.csv",
    index=False
)

Final selected sliding-window ESM model: Soft Voting


,9
model,Soft Voting
dataset,validation
accuracy,0.659341
precision,0.666667
recall_sensitivity,0.642336
specificity,0.676471
f1,0.654275
roc_auc,0.728532
pr_auc,0.730087
mcc,0.318978


In [36]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

sw_final_test_metrics = evaluate_binary_classifier(
    model=final_sw_model,
    X=X_test_sw,
    y=y_test,
    model_name=final_sw_model_name,
    dataset_name="test"
)

sw_final_test_metrics["representation"] = REPRESENTATION_NAME

sw_final_test_metrics_df = pd.DataFrame([sw_final_test_metrics])

display(sw_final_test_metrics_df)

sw_final_test_metrics_df.to_csv(
    RESULT_DIR / "esm2_sw_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,test,0.582418,0.578571,0.595588,0.569343,0.586957,0.627684,0.627788,0.164984,78,59,55,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [37]:
# ============================================================
# DIAGNOSTIC TEST ALL MODELS
# Not used for model selection.
# ============================================================

sw_diagnostic_test_records = []

for model_name, model in sw_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_sw,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )

    metrics["representation"] = REPRESENTATION_NAME
    sw_diagnostic_test_records.append(metrics)

sw_diagnostic_test_df = pd.DataFrame(sw_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(sw_diagnostic_test_df)

sw_diagnostic_test_df.to_csv(
    RESULT_DIR / "esm2_sw_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,test_diagnostic,0.589744,0.582192,0.625000,0.554745,0.602837,0.639545,0.624807,0.180180,76,61,51,85,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
4,Soft Voting,test_diagnostic,0.582418,0.578571,0.595588,0.569343,0.586957,0.627684,0.627788,0.164984,78,59,55,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
5,Stacking,test_diagnostic,0.586081,0.582734,0.595588,0.576642,0.589091,0.625322,0.626383,0.172258,79,58,55,81,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
3,LightGBM,test_diagnostic,0.567766,0.564286,0.580882,0.554745,0.572464,0.615554,0.607315,0.135671,76,61,57,79,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
1,SVM RBF,test_diagnostic,0.582418,0.584615,0.558824,0.605839,0.571429,0.614722,0.619246,0.164849,83,54,60,76,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
2,Random Forest,test_diagnostic,0.571429,0.561290,0.639706,0.503650,0.597938,0.614105,0.605149,0.144690,69,68,49,87,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022


In [38]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_sw_model.predict(X_test_sw)
y_test_score = get_positive_class_score(final_sw_model, X_test_sw)

sw_test_predictions_df = meta_test_sw.copy()
sw_test_predictions_df["true_label"] = y_test.values
sw_test_predictions_df["pred_label"] = y_test_pred
sw_test_predictions_df["pred_score_t2d_associated"] = y_test_score

display(sw_test_predictions_df.head())

sw_test_predictions_df.to_csv(
    RESULT_DIR / "esm2_sw_final_test_predictions_v1.csv",
    index=False
)

,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation,true_label,pred_label,pred_score_t2d_associated
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,0,1,0.685409
1,1,ENSG00000123080,CDKN2C,P42773,1,168,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,1,0,0.428244
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,1,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,0,1,0.760105
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,3,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,0,0,0.492170
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,2,1022,1022,ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022,1,1,0.573962


In [39]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in sw_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")

    model_path = MODEL_DIR / f"esm2_sw_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_sw_model_path = MODEL_DIR / "esm2_sw_final_selected_model_v1.pkl"
joblib.dump(final_sw_model, final_sw_model_path)

print("Final sliding-window model saved:", final_sw_model_path)

Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_logistic_regression_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_svm_rbf_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_random_forest_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_lightgbm_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_soft_voting_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_stacking_best_estimator_v1.pkl
Final sliding-window model saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_sliding_window_embedding/models/esm2_sw_final_selected_model_v1.pkl